# 🏛️ Architecture Médaillon : Vue d'Ensemble & Tests d'Environnement

Bienvenue dans ce projet pédagogique complet dédié à l'**Architecture Médaillon** (Data Lakehouse).

---

## 🎯 Objectifs de ce Notebook
1. Comprendre les fondements théoriques de l'architecture Médaillon (Bronze ➔ Silver ➔ Gold).
2. Distinguer le rôle d'un **Data Lake** (MinIO) de celui d'un **Data Warehouse** (PostgreSQL).
3. Vérifier la connectivité et la santé de notre infrastructure conteneurisée avec Docker.
4. Initialiser une session **Apache Spark** configurée pour communiquer avec le stockage objet S3 local.


---
## 1. Rappel Théorique : L'Architecture Médaillon

L'architecture Médaillon structure les données d'une entreprise en trois niveaux de raffinement successifs :

| Couche | Nom | Rôle & Caractéristiques | Format & Moteur |
| :--- | :--- | :--- | :--- |
| 🥉 **Bronze** | *Raw / Landing* | Données brutes telles quelles, immutables, avec historique complet et métadonnées d'ingestion. | Parquet / JSON / CSV sur MinIO (S3) |
| 🥈 **Silver** | *Cleaned / Conformed* | Données nettoyées, typées, dédupliquées, enrichies et filtrées des anomalies. | Parquet partitionné sur MinIO (S3) |
| 🥇 **Gold** | *Curated / Business* | Données agrégées, modélisées en étoile (faits/dimensions) et optimisées pour la BI. | Tables & Vues SQL dans PostgreSQL |

```
[ AWS S3 Open Data ]
         │
         ▼ (Ingestion brute)
  ┌──────────────┐
  │ Couche BRONZE │  MinIO S3 (Bucket 'bronze')
  └──────┬───────┘
         │
         ▼ (Nettoyage & Enrichissement PySpark)
  ┌──────────────┐
  │ Couche SILVER │  MinIO S3 (Bucket 'silver' Parquet partitionné)
  └──────┬───────┘
         │
         ▼ (Agrégations Métier & Modélisation)
  ┌──────────────┐
  │  Couche GOLD  │  PostgreSQL (Data Marts & Vues analytiques)
  └──────┬───────┘
         │
         ▼ (Visualisation décisionnelle)
  ┌──────────────┐
  │   METABASE   │  Dashboards & KPIs Business
  └──────────────┘
```


---
## 2. Vérification des Variables d'Environnement


In [ ]:
import os
from pprint import pprint

print("=== Variables d'environnement détectées ===")
env_keys = [
    "MINIO_ENDPOINT", "MINIO_ACCESS_KEY", "MINIO_SECRET_KEY",
    "POSTGRES_HOST", "POSTGRES_PORT", "POSTGRES_DB", "POSTGRES_USER"
]
for k in env_keys:
    val = os.getenv(k, "NON DÉFINI (valeur par défaut)")
    if "SECRET" in k or "PASSWORD" in k:
        val = "********"
    print(f"{k:20} : {val}")


---
## 3. Test de Connexion au Stockage Objet (MinIO S3)

Nous utilisons la bibliothèque standard `boto3` pour vérifier que le service MinIO répond et que les buckets `bronze` et `silver` sont prêts.


In [ ]:
import boto3

s3_client = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"),
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "admin"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "password123"),
    region_name="us-east-1"
)

try:
    response = s3_client.list_buckets()
    buckets = [b["Name"] for b in response.get("Buckets", [])]
    print("✅ Connexion à MinIO réussie !")
    print(f"📦 Buckets disponibles : {buckets}")
    
    for required_bucket in ["bronze", "silver"]:
        if required_bucket in buckets:
            print(f"   ✓ Bucket '{required_bucket}' est prêt.")
        else:
            print(f"   ⚠️ Bucket '{required_bucket}' manquant, création...")
            s3_client.create_bucket(Bucket=required_bucket)
            print(f"   ✓ Bucket '{required_bucket}' créé avec succès.")
except Exception as e:
    print(f"❌ Erreur de connexion à MinIO : {e}")


---
## 4. Test de Connexion au Data Warehouse (PostgreSQL)

Vérifions que PostgreSQL est accessible et testons une requête simple.


In [ ]:
from sqlalchemy import create_engine, text

pg_host = os.getenv("POSTGRES_HOST", "postgres")
pg_port = os.getenv("POSTGRES_PORT", "5432")
pg_db = os.getenv("POSTGRES_DB", "gold_db")
pg_user = os.getenv("POSTGRES_USER", "postgres")
pg_pwd = os.getenv("POSTGRES_PASSWORD", "postgres123")

conn_str = f"postgresql+psycopg2://{pg_user}:{pg_pwd}@{pg_host}:{pg_port}/{pg_db}"

try:
    engine = create_engine(conn_str)
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();")).fetchone()
        print("✅ Connexion à PostgreSQL réussie !")
        print(f"🐘 Version : {result[0]}")
        
        schema_check = conn.execute(
            text("SELECT schema_name FROM information_schema.schemata WHERE schema_name = 'gold';")
        ).fetchone()
        if schema_check:
            print("   ✓ Schéma 'gold' présent.")
        else:
            print("   ⚠️ Schéma 'gold' absent, création en cours...")
            conn.execute(text("CREATE SCHEMA IF NOT EXISTS gold;"))
            conn.commit()
            print("   ✓ Schéma 'gold' créé.")
except Exception as e:
    print(f"❌ Erreur de connexion à PostgreSQL : {e}")


---
## 5. Démarrage et Test d'Apache Spark (PySpark avec support S3A)

Pour que Spark puisse lire et écrire dans MinIO, nous configurons le connecteur `S3A`.
Vérifions le bon fonctionnement du moteur distribué.


In [ ]:
from pyspark.sql import SparkSession

minio_endpoint = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.getenv("MINIO_ACCESS_KEY", "admin")
minio_secret_key = os.getenv("MINIO_SECRET_KEY", "password123")

spark = (
    SparkSession.builder
    .appName("00_Architecture_Test")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Session PySpark initialisée avec succès !")
print(f"🔥 Version de Spark : {spark.version}")


### Test d'écriture et de lecture Spark vers MinIO (protocole `s3a://`)


In [ ]:
# Création d'un petit DataFrame de test
test_data = [("Bronze", 1, "Raw Data"), ("Silver", 2, "Cleaned Data"), ("Gold", 3, "Business Ready")]
columns = ["Layer", "Priority", "Description"]

df_test = spark.createDataFrame(test_data, columns)
print("DataFrame de test :")
df_test.show()

# Écriture dans MinIO bucket 'bronze'
test_path = "s3a://bronze/_test_connection/medallion_layers.parquet"
df_test.write.mode("overwrite").parquet(test_path)
print(f"✓ Écrit avec succès vers {test_path}")

# Relecture depuis MinIO
df_read = spark.read.parquet(test_path)
print("Relecture depuis MinIO :")
df_read.show()
print("🎉 Test d'intégration MinIO + PySpark 100% fonctionnel !")


---
## 🏁 Conclusion du Notebook 00
Tous les services sont opérationnels :
- **MinIO** est accessible et prêt à recevoir les couches Bronze & Silver.
- **PostgreSQL** est prêt pour la couche Gold.
- **Apache Spark** communique sans encombre via le connecteur S3A.

👉 Passez au notebook suivant : `01_ingestion_bronze.ipynb` pour démarrer l'ingestion massive des données ouvertes d'AWS.
